# Stage 3: Fractional Differentiation (corrected)

AFML Ch 5. We sweep `d` ∈ [0, 1] in 0.05 steps and choose the smallest `d` such that the FFD series is **stationary** (ADF p-value < 0.05) while still **preserving memory** (correlation with the aligned log-price ≥ 0.9). If no `d` satisfies both, we fall back to the stationary `d` with the highest correlation and document why.

_This notebook was regenerated after the FFD weight-direction bug was fixed in `src/fracdiff.py`._

In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

sys.path.insert(0, os.path.abspath('../..'))
from src.fracdiff import (
    get_weights_ffd, frac_diff_ffd, find_min_d, plot_min_ffd,
)

plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('../../reports/figures', exist_ok=True)

## 1. Load log price

In [ ]:
close_df = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
log_close = np.log(close_df['Adj Close']).rename('log_close').dropna()
print(f'log_close: {len(log_close)} obs, '
      f'{log_close.index.min().date()} → {log_close.index.max().date()}')

## 2. Sweep d ∈ [0, 1] in 0.05 steps

For every d we compute the FFD series, run the ADF test on it, and measure correlation with the aligned original log-price. We also record how many observations survive (the FFD window forces us to drop the first `window_length − 1` rows).

In [ ]:
selection = find_min_d(
    log_close,
    d_range=np.arange(0.0, 1.05, 0.05),
    threshold=1e-5,
    p_threshold=0.05,
    corr_threshold=0.9,
)
sweep = selection['sweep'].copy()
sweep[['adf_stat', 'p_value', 'correlation']] = (
    sweep[['adf_stat', 'p_value', 'correlation']].astype(float).round(4)
)
sweep

## 3. Selected d*

In [ ]:
d_star = selection['d_star']
row = selection['sweep'].loc[d_star]

print(f'rule applied:   {selection["rule_applied"]}')
print(f'strict (p<{selection["p_threshold"]} AND '
      f'corr>={selection["corr_threshold"]}) satisfied: '
      f'{selection["strict_satisfied"]}')
print()
print(f'd*               = {d_star:.2f}')
print(f'  ADF statistic  = {row["adf_stat"]:.4f}')
print(f'  ADF p-value    = {row["p_value"]:.6f}')
print(f'  correlation    = {row["correlation"]:.4f}')
print(f'  n_obs retained = {int(row["n_obs"])}')
print(f'  window length  = {int(row["window_length"])} weights')

### Why this `d*`?

Reading the table above:
- For very small `d` (≈ 0), the series is essentially the original   non-stationary log price (high ADF p-value, correlation ≈ 1).
- For `d` close to 1, we approach first differences (returns), which   are stationary but discard the long-memory level information   (correlation with log-price drops sharply).
- The selection rule walks from small `d` upward and stops at the   first value that **rejects the unit root** while still keeping   most of the level information. If the strict thresholds cannot   both be met, the fallback prefers a stationary series with   maximum correlation, since the methodological cost of forfeiting   memory is higher than the cost of a less-tight ADF p-value.
- The `window_length` column is also informative: large windows for   small `d` cost us many leading observations, which is the price   of preserving memory.

## 4. P9 — ADF / correlation dual-axis plot

In [ ]:
fig9, _ = plot_min_ffd(
    log_close,
    d_range=np.arange(0.0, 1.05, 0.05),
    threshold=1e-5,
    p_threshold=0.05,
    corr_threshold=0.9,
)
fig9.savefig(os.path.join(FIG_DIR, 'P9_fracdiff_adf_correlation.png'),
             dpi=150, bbox_inches='tight')
plt.show()

## 5. P10 — Original vs first-diff vs d* overlay

In [ ]:
fd_dstar = frac_diff_ffd(log_close, d=d_star, threshold=1e-5)
fd_d1    = frac_diff_ffd(log_close, d=1.0,   threshold=1e-5)

fig10, ax = plt.subplots(figsize=(14, 6))
ax.plot(log_close.index, log_close, color='black', alpha=0.35,
        label='original log price')

ax2 = ax.twinx()
ax2.plot(fd_d1.index, fd_d1, color='#e34a33', alpha=0.45,
         label='d = 1 (first differences)')
ax2.plot(fd_dstar.index, fd_dstar, color='#2c7fb8', alpha=0.9,
         label=f'd* = {d_star:.2f} (FFD)')

ax.set_ylabel('log price', color='black')
ax2.set_ylabel('differentiated value', color='#2c7fb8')
ax.set_title(
    'Original vs first-diff vs FFD(d*)  —  '
    f'corr(d*, log_close) = {fd_dstar.corr(log_close.loc[fd_dstar.index]):.3f}'
)

lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2, loc='upper left')
fig10.tight_layout()
fig10.savefig(os.path.join(FIG_DIR, 'P10_fracdiff_overlay.png'),
              dpi=150, bbox_inches='tight')
plt.show()

## 6. Save the FFD series at d\*

In [ ]:
out = fd_dstar.to_frame(name='fracdiff_d_star')
out.attrs = {
    'd_star':         float(d_star),
    'rule_applied':   selection['rule_applied'],
    'p_threshold':    float(selection['p_threshold']),
    'corr_threshold': float(selection['corr_threshold']),
}
out.to_parquet(f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet')
print(f"Saved ../data/processed/nvda_fracdiff.parquet  "
      f"({len(out)} rows, single column 'fracdiff_d_star')")

## 7. Notes

- **Bug fixed in `src/fracdiff.py`**: `frac_diff_ffd` previously   passed the oldest-first weights array directly into `np.convolve`,   which (because of `np.convolve`'s built-in kernel flip) ended up   applying `w₀ = 1` to the *oldest* sample of each window instead of   the most recent. The fix reverses the weights once before the   convolution so that `X̃_t = Σ w_k · X_{t-k}` is computed correctly.
- **Index alignment hardened**: `frac_diff_ffd` now derives its   output index from the cleaned (forward-filled, dropna'd) series   rather than from the raw input — so a future caller with NaNs in   the raw series will not silently misalign output rows.
- **Selection rule**: `find_min_d` now requires both ADF p-value <   0.05 AND correlation ≥ 0.9, with a documented fallback to the   stationary d with maximum correlation when both can't be   satisfied.
- **Downstream impact**: `nvda_features.parquet` and   `nvda_modelling_dataset.parquet` still embed the *old* fracdiff   column. Re-run notebook 05 to refresh them; that will in turn   invalidate Stage 4/5/6 artefacts.